In [ ]:
!pip3 install selenium webdriver-manager pandas openpyxl

In [ ]:
!pip3 install scheduler

In [ ]:
import pandas as pd
import openpyxl
from typing import Optional

def _get_link_if_exists(cell) -> Optional[str]:
    try:
        return cell.hyperlink.target
    except AttributeError:
        return None


def extract_hyperlinks_from_xlsx(file_path, sheet_name=None):
    wb = load_workbook(filename=file_path, data_only=True)

    # 👉 Nếu không truyền sheet_name hoặc sheet không tồn tại → lấy sheet đầu tiên
    if not sheet_name or sheet_name not in wb.sheetnames:
        ws = wb.worksheets[0]
        print(f"⚠️ Sheet '{sheet_name}' không tồn tại → dùng sheet mặc định: '{ws.title}'")
    else:
        ws = wb[sheet_name]

    urls = []
    hotel_names = []
    room_types = []

    for row in ws.iter_rows(min_row=2, max_col=2):
        hotel_cell = row[0]
        room_cell = row[1]

        hotel_names.append(hotel_cell.value)
        urls.append(hotel_cell.hyperlink.target if hotel_cell.hyperlink else "")
        room_types.append(room_cell.value)

    return pd.DataFrame({
        "Hotel": urls,
        "Hotel_name": hotel_names,
        "Room_type": room_types
    })


In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time
from datetime import datetime, timedelta, date
from openpyxl import load_workbook
import os

# ======= HÀM LẤY LINK TỪ EXCEL ========
def extract_hyperlinks_from_xlsx(file_path, sheet_name):
    wb = load_workbook(filename=file_path, data_only=True)
    ws = wb[sheet_name]

    urls = []
    hotel_names = []
    room_types = []

    for row in ws.iter_rows(min_row=2, max_col=2):
        hotel_value = row[0]
        room_type = row[1]

        hotel_names.append(hotel_value.value)
        if hotel_value.hyperlink:
            urls.append(hotel_value.hyperlink.target)
        else:
            urls.append("")

        room_types.append(room_type.value)

    df = pd.DataFrame({"Hotel": urls, "Hotel_name": hotel_names, 'Room_type': room_types})
    return df

# ======= HÀM SET NGÀY CÓ XỬ LÝ NÚT NEXT ========
def set_dates(driver, wait, checkin_date, checkout_date):
    def open_calendar():
        checkin_box = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "[data-selenium='checkInBox']")))
        checkin_box.click()
        time.sleep(1)

    def click_next_month_until_visible(target_date):
        for _ in range(12):  # tránh lặp vô hạn
            try:
                selector = f"[data-selenium-date='{target_date.strftime('%Y-%m-%d')}']"
                driver.find_element(By.CSS_SELECTOR, selector)
                return  # nếu tìm thấy thì dừng
            except:
                try:
                    next_btn = driver.find_element(By.CSS_SELECTOR, "[data-selenium='calendar-next-month-button']")
                    next_btn.click()
                    time.sleep(1)
                except Exception as e:
                    print("❌ Không tìm được nút next tháng:", e)
                    break

    def select_date(date):
        click_next_month_until_visible(date)
        selector = f"[data-selenium-date='{date.strftime('%Y-%m-%d')}']"
        date_button = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, selector)))
        driver.execute_script("arguments[0].scrollIntoView(true);", date_button)
        date_button.click()
        time.sleep(1)

    try:
        open_calendar()
        select_date(checkin_date)
        select_date(checkout_date)
    except Exception as e:
        print("❌ Lỗi chọn ngày:", e)

# ======= HÀM CÀO GIÁ ========
def scrape_room_prices(url, check_in_date_str: str, check_out_date_str: str, room_type_name=""):
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36")
    
    driver = webdriver.Chrome(service=ChromeService(ChromeDriverManager().install()), options=options)
    driver.get(url)
    wait = WebDriverWait(driver, 30)
    results = []

    try:
        time.sleep(2)
        try:
            close_btn = driver.find_element(By.CSS_SELECTOR, ".ab-close-button")
            close_btn.click()
        except:
            pass

        checkin_date = date.fromisoformat(check_in_date_str)
        checkout_date = date.fromisoformat(check_out_date_str)

        if checkin_date >= checkout_date:
            raise Exception("Check out date can't be earlier than check in date")

        set_dates(driver, wait, checkin_date, checkout_date)

        search_btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "[data-selenium='searchButton']")))
        search_btn.click()

        time.sleep(2)
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div#roomGrid")))
        time.sleep(1)
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)

        room_cards = driver.find_elements(By.CSS_SELECTOR, "div[data-selenium='MasterRoom']")
        for card in room_cards:
            try:
                name = card.find_element(By.CSS_SELECTOR, "[data-selenium='masterroom-title-name']").text
            except:
                name = "NA"
            try:
                price = card.find_element(By.CSS_SELECTOR, "[data-selenium='PriceDisplay']").text
            except:
                price = "NA"
            results.append({"room": name, "price": price})

    except Exception as e:
        print(f"❌ Lỗi khi cào: {e}")
        with open("agoda_debug.html", "w", encoding="utf-8") as f:
            f.write(driver.page_source)

    finally:
        driver.quit()

    return results

# ======= MAIN ========
if __name__ == "__main__":
    df_urls = extract_hyperlinks_from_xlsx("./Competitor Tracking.xlsx", "Hotel Link")

    backup_filename = "hotel_prices_temp.xlsx"
    if os.path.exists(backup_filename):
        print(f"📂 Đang đọc file tạm: {backup_filename}")
        df_prev = pd.read_excel(backup_filename)
    else:
        df_prev = pd.DataFrame(columns=["Hotel", "Room", "Price W1", "Price W2", "Price W3", "Price W4", "Price W5", "Price W6"])

    prev_data = {}
    for _, row in df_prev.iterrows():
        key = (row["Hotel"], row["Room"])
        prev_data[key] = {f"Price W{i}": row.get(f"Price W{i}", "NA") for i in range(1, 7)}

    result_hotel_price = {
        "Hotel": [],
        "Room": [],
        "Price W1": [],
        "Price W2": [],
        "Price W3": [],
        "Price W4": [],
        "Price W5": [],
        "Price W6": [],
    }

    # ✅ Cố định ngày gốc ngay tại thời điểm bắt đầu script
    base_checkin = datetime.today().replace(hour=0, minute=0, second=0, microsecond=0) + timedelta(days=1)
    base_checkout = base_checkin + timedelta(days=1)

    week_offsets = [0, 7, 14, 21, 28, 35]
    all_week_prices = {}

    for week_num, offset in enumerate(week_offsets, start=1):
        key_prefix = f"Price W{week_num}"

        for index, row in df_urls.iterrows():
            hotel_name = row["Hotel_name"]
            hotel_url = row["Hotel"]
            room_type = row["Room_type"]
            key = (hotel_name, room_type)

            if not hotel_url.startswith("http"):
                print(f"❌ Bỏ qua do URL không hợp lệ: {hotel_url}")
                continue

            if key in prev_data and key_prefix in prev_data[key] and prev_data[key][key_prefix] != "NA":
                print(f"✅ Đã có {key_prefix} cho {hotel_name} - {room_type}, bỏ qua")
                if key not in all_week_prices:
                    all_week_prices[key] = {}
                all_week_prices[key][key_prefix] = prev_data[key][key_prefix]
                continue

            already_collected = set()

            for retry_offset in range(0, 7):
                checkin = base_checkin + timedelta(days=offset + retry_offset)
                checkout = base_checkout + timedelta(days=offset + retry_offset)
                checkin_str = checkin.strftime("%Y-%m-%d")
                checkout_str = checkout.strftime("%Y-%m-%d")

                print(f"⏳ Crawling {key_prefix} for {hotel_name} with {room_type} | Try +{retry_offset} | {checkin_str} → {checkout_str}")

                data = scrape_room_prices(hotel_url, checkin_str, checkout_str)

                rooms_this_round = 0
                for room_data in data:
                    room_name = room_data["room"]

                    if room_name.strip() != room_type.strip():
                        continue

                    if key not in all_week_prices:
                        all_week_prices[key] = {}

                    if key_prefix in all_week_prices[key]:
                        already_collected.add(key)
                        continue

                    price = room_data["price"]
                    if price != "NA":
                        all_week_prices[key][key_prefix] = price
                        already_collected.add(key)
                        rooms_this_round += 1

                if (key in already_collected) or (key_prefix in all_week_prices.get(key, {})):
                    break

            if key_prefix not in all_week_prices.get(key, {}):
                if key not in all_week_prices:
                    all_week_prices[key] = {}
                all_week_prices[key][key_prefix] = "NA"

            # 🔃 Lưu tạm sau mỗi khách sạn / tuần
            temp_result = {
                "Hotel": [],
                "Room": [],
                "Price W1": [],
                "Price W2": [],
                "Price W3": [],
                "Price W4": [],
                "Price W5": [],
                "Price W6": [],
            }

            for (hotel, room), prices in all_week_prices.items():
                temp_result["Hotel"].append(hotel)
                temp_result["Room"].append(room)
                for i in range(1, 7):
                    temp_result[f"Price W{i}"].append(prices.get(f"Price W{i}", "NA"))

            df_temp = pd.DataFrame(temp_result)
            df_temp.to_excel(backup_filename, index=False)
            print(f"💾 Đã lưu tạm thời sau: {hotel_name} - {room_type} | {key_prefix}")

    # Xuất file chính thức
    for (hotel, room), prices in all_week_prices.items():
        result_hotel_price["Hotel"].append(hotel)
        result_hotel_price["Room"].append(room)
        for i in range(1, 7):
            result_hotel_price[f"Price W{i}"].append(prices.get(f"Price W{i}", "NA"))

    df = pd.DataFrame(result_hotel_price)
    final_filename = f"hotel_prices_{datetime.today().strftime('%Y%m%d')}.xlsx"
    df.to_excel(final_filename, index=False)
    print(f"\n✅ Saved to: {final_filename}")


❌ Lỗi khi cào: Message: 
Stacktrace:
0   chromedriver                        0x00000001008c3928 cxxbridge1$str$ptr + 3098388
1   chromedriver                        0x00000001008bbc04 cxxbridge1$str$ptr + 3066352
2   chromedriver                        0x000000010039e83c _RNvCs5DBLTqoOdVp_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 75008
3   chromedriver                        0x00000001003e7d7c _RNvCs5DBLTqoOdVp_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 375360
4   chromedriver                        0x0000000100426fe8 _RNvCs5DBLTqoOdVp_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 634028
5   chromedriver                        0x00000001003dc07c _RNvCs5DBLTqoOdVp_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 326976
6   chromedriver                        0x0000000100882884 cxxbridge1$str$ptr + 2831984
7   chromedriver                        0x0000000100885fcc cxxbridge1$str$ptr + 2846136
8   chromedriver                        0x0000000100867578 cxxbridge1$str$pt

KeyboardInterrupt: 